# 第12回 世界の気候を、軸で読む


---


## 前回からの接続


第11回は、顔の4,096個の数を、少ない向きで持ちました。向きの意味は、両端の顔を見て読みました。今日は、列に名前（何月の気温か、何月の雨か）があるデータで、向きの意味を読みます。


---


## 今日の分析目標


世界の都市の気候（24個の数）を少ない軸にまとめると、その軸は何を表すか。地図の上で都市はどう並ぶかに答える。

まず、気象のAPIが返したJSONを読み、67都市×24列の表にします。次に、標準化（standardization）しないときとしたときで、主成分分析（principal component analysis）の向きを比べます。そして、負荷量（loading）を月の折れ線で描いて、軸に名前を付けます。さらに、得点の地図に都市を並べます。最後に、その地図がどこまで元のデータを写しているかを確かめます。

もう1つのねらいは、APIが返すJSONを、辞書のキーとリストの番号でたどって、表に直せるようになることです。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

世界の気象のデータが、APIという窓口から誰でも取り出せる形で公開されています。場所と期間を書いて頼むと、JSONという形式の文字で返ってきます。今日は、Open-Meteo（https://open-meteo.com/ 、CC BY 4.0）から取りました。

たとえば東京なら、次のように頼みます（緯度・経度、期間、ほしい値）。

```text
https://archive-api.open-meteo.com/v1/archive?latitude=35.68&longitude=139.69&start_date=2015-01-01&end_date=2024-12-31&daily=temperature_2m_mean,precipitation_sum
```

これを67都市ぶん頼み、返ってきたJSONを1つのファイルにまとめました。


### 場面

気候は都市ごとに違います。東京は夏に暑く雨が多く、シンガポールは1年中暑く、シドニーは季節が逆で、ヤクーツクは1月に−36.6°Cになります。

1都市の気候は、月ごとの気温12個と降水量12個、合わせて24個の数です。24個を少ない軸にまとめて都市を地図に並べ、その軸が何を表すのかを読みます。


### 手元のデータ

Open-Meteoの過去の気象APIから、2015〜2024年の毎日の値を取りました。67都市（北半球50、南半球17）で、1都市あたり3,653日ぶんの平均気温（°C）と降水量（mm）です。中身は表ではなく、辞書とリストが入れ子になっています。

使うときは、"Weather data by Open-Meteo.com"のクレジットを書きます。


### 最後に出す答え

24個の数を少ない軸にまとめると、それぞれの軸は何を表すか。地図の上で、都市はどう並ぶか。その地図は、どこまで信じてよいかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. JSONを読む：入れ子のJSONから、67都市×24列の表を作ります。
2. 単位で変わる？：えっ、標準化しないと、第1の向きは降水量ばかりです。気温がほとんど入りません。
3. 軸を読む：標準化すると、第1は1年中の気温（暑さ）、第2は12〜4月の雨、第3は6〜9月の雨と読めます。
4. 地図を読む：緯度を使っていないのに、南半球の都市が地図の上に集まります。
5. 何本残す？：2本の地図で、24列の散らばりの66%。4本なら93%です。

このあと、この5つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| JSONを読む（1節） | APIのデータはどんな形？ | キーと番号をたどって1都市の毎日の値を取り出し、月ごとに直して67都市の表を作る |
| 単位で変わる？（2節） | 単位で答えは変わるか？ | 標準化しないときとしたときで、主成分分析の向きを比べる |
| 軸を読む（3節） | 軸は何を表す？ | 負荷量を月の折れ線で描き、名前を付ける |
| 地図を読む（4節） | 都市はどこに並ぶ？ | 得点の地図に都市名を付け、南半球と比べる |
| 何本残す？（5節） | 地図はどこまで信じてよい？ | 累積寄与率と、地図の距離が元の距離をどれだけ写すかを見る |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- JSONを、辞書のキーとリストの番号でたどり、表に直せる
- 単位がまちまちな列で、標準化しないと主成分分析の答えが単位で変わることを説明できる
- 負荷量を列の並びに沿って描き、軸に名前を付けられる
- 得点の地図を描いて都市の並びを読み、使わなかった情報で名前を確かめられる
- 累積寄与率（cumulative proportion of variance explained）と距離の写り方で、地図をどこまで信じてよいかを確かめられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


---


## 1. APIのデータはどんな形？


ここから、段1「JSONを読む」（1節）です。問いは、APIが返したJSONは、どんな形をしていて、どう表に直せばよいか、です。


### JSONとは

辞書（`{キー: 値}`）とリスト（`[…]`）を、入れ子にして書く文字の形式です。辞書の値に、リストや別の辞書を入れてかまいません。読むと、Pythonの辞書とリストになります。


### おさらい：辞書とリスト


In [ ]:
d = {'name': '東京', 'temps': [4.1, 5.3, 9.4]}
print(d['name'])
print(d['temps'][1])


辞書はキーで（第5回）、リストは番号で（第4回）取り出します。`d['temps'][1]`は、たどる順に2回取り出しています。


### お手本：JSONを読む

- 役割：JSONのファイルを読み、いちばん外側のキーと期間を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`world_climate.json`
- 出力：`data`（外側の辞書をキーで引ける形にしたもの）


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
data = pd.read_json(f'{DATA_DIR}/world_climate.json', typ='series')
print(data.index.tolist())
print(data['period'])


#### 1行ずつ読む

- `DATA_DIR`の行：データの置き場所を`DATA_DIR`にしまう（Colabでは、置き場所のURLが2行に分かれて入っている）
- `pd.read_json`の行：JSONを読む。`typ='series'`で、いちばん外側の辞書を、キーで引ける形にする。中のリストと辞書は、Pythonのリストと辞書のまま
- `print`の行：外側のキーの一覧と、`'period'`の値（期間）を見る


### お手本：都市のリスト

- 役割：`'cities'`のリストから、0番目の都市の辞書を取り出す
- 入力：`data`
- 出力：辞書`city`


In [ ]:
print(len(data['cities']))
city = data['cities'][0]
print(city.keys())
print(city['name'], city['country'])


#### 1行ずつ読む

- 1行目：`'cities'`の値はリスト。67都市が並ぶ
- 2行目：リストの0番目（東京）を取り出す。これは辞書
- 3・4行目：その辞書のキーの一覧と、名前・国を見る


### お手本：APIの返事をたどる

- 役割：APIが返したそのままの辞書から、緯度・経度と毎日の気温を取り出す
- 入力：辞書`city`
- 出力：辞書`res`と表示


In [ ]:
res = city['response']
print(res['latitude'], res['longitude'])
print(res['daily'].keys())
print(res['daily']['temperature_2m_mean'][:5])


#### 1行ずつ読む

- 1行目：APIが返したそのままの辞書を`res`にする
- 2行目：APIが使った緯度・経度
- 3行目：`'daily'`の中は辞書。キーの一覧を見る
- 4行目：その中の気温はリストで、先頭5日の値を見る

辞書 → リスト → 辞書 → 辞書 → リストの入れ子です。キーと番号を、外から順にたどります。


#### 中身を見る：1回ずつたどる

1段ずつ取り出して、それぞれが辞書かリストかを確かめます（`type`）。


In [ ]:
a = data['cities']
b = a[0]
c = b['response']
e = c['daily']
f = e['precipitation_sum']
print(type(a), type(b), type(c), type(e), type(f))
print(len(a), len(f))


1段たどるごとに、リストと辞書が入れ替わります。いちばん奥は、3,653日ぶんのリストです。


> 今日の重要概念（Python）：入れ子の辞書とリスト
>
> JSONは、辞書とリストの入れ子。キーと番号を、外から順にたどると、欲しい値に届く。たどるたびに、辞書かリストかを確かめる。


### お手本：1都市を月ごとに

1都市は、3,653日ぶんの気温と降水量です。欲しいのは、月ごとの平均気温（12個）と、月ごとの降水量（12個）です。降水量は、10年ぶんの各月の合計を10で割り、1年あたりにします。

- 役割：1都市の毎日の値を表にし、月ごとの平均気温と降水量を出す
- 入力：辞書`city`
- 出力：`temp`・`rain`（12個ずつ）


In [ ]:
daily = pd.DataFrame(city['response']['daily'])
daily['月'] = pd.to_datetime(daily['time']).dt.month
temp = daily.groupby('月')['temperature_2m_mean'].mean()
rain = daily.groupby('月')['precipitation_sum'].sum() / 10
print(temp.round(1).tolist())
print(rain.round(0).tolist())


#### 1行ずつ読む

- 1行目：`'daily'`の辞書（キーが列の名前、値がリスト）を、そのまま表にする
- 2行目：日付の文字を日付に直し（`pd.to_datetime`）、月の番号の列を作る
- 3・4行目：月ごとに、気温は平均、降水量は合計÷10（`groupby`、第6回）
- 5・6行目：12個ずつを表示する


#### 中身を見る：表になった毎日の値


In [ ]:
daily.head(3)


辞書のキー（`time`など）が列の名前、リストの中身が列の値になりました。


### お手本：1都市ぶんを関数に

67都市に同じ手順を当てるので、関数にします（第7回）。

- 役割：都市の辞書を受け取り、24個の数のリストを返す
- 入力：都市1つの辞書`city`
- 出力：気温12個＋降水量12個のリスト


In [ ]:
def climate_row(city):
    daily = pd.DataFrame(city['response']['daily'])
    daily['月'] = pd.to_datetime(daily['time']).dt.month
    temp = daily.groupby('月')['temperature_2m_mean'].mean()
    rain = daily.groupby('月')['precipitation_sum'].sum() / 10
    return list(temp.round(1)) + list(rain.round(0))


#### 1行ずつ読む

- 1行目：関数`climate_row`を作る。入力は都市1つの辞書`city`
- 2〜5行目：上のお手本と同じ
- 6行目：気温12個と降水量12個をリストにして、つないで（`+`）返す


### お手本：67都市の表

- 役割：67都市それぞれに`climate_row`を当て、行の名前と列の名前の付いた表にする
- 入力：`data['cities']`
- 出力：表`X`（67行×24列）


In [ ]:
rows = []
names = []
for city in data['cities']:
    rows.append(climate_row(city))
    names.append(city['name'])
cols = []
for m in range(1, 13):
    cols.append('気温' + str(m) + '月')
for m in range(1, 13):
    cols.append('降水' + str(m) + '月')
X = pd.DataFrame(rows, index=names, columns=cols)
print(X.shape)


#### 1行ずつ読む

- 1・2行目：24個のリストをためるリストと、都市名をためるリストを作る
- 3行目：都市のリストから、都市の辞書を1つずつ取る
- 4・5行目：24個の数と名前を、それぞれの後ろに足す
- 6〜10行目：列の名前を24個作る（`str(m)`で数を文字に直してつなぐ）
- 11行目：リストのリストを、行の名前と列の名前付きの表にする
- 12行目：67都市×24列


#### 中身を見る：表


In [ ]:
X.iloc[:3, [0, 6, 12, 18]]


1行が1都市、列が月ごとの気温と降水量です。ここでは1月と7月だけを見ました。


### 結果を見る前に、予想を書く

24列を少ない軸にまとめたら、いちばん大きい向きは何になるでしょうか。予想と、その理由を1行で書いてください。


（ここに予想を書く）


### TODO①：都市の番号を替える（予想 → 実行 → 比べる）

下のセルは、都市のリストから1つ取り出し、名前・緯度と、月ごとの気温12個を表示します。

まず、予想を書いてください。62番の都市の、月ごとの気温はどんな形でしょうか（何月が暖かいでしょうか）。


（ここに予想を書く）


予想を書いたら、セルの`0`を`62`に替えて実行してください。


In [ ]:
# TODO: 0 を 62 に替えて実行する
city = data['cities'][0]
print(city['name'], city['response']['latitude'])
print(climate_row(city)[:12])


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。何月が暖かく、何月が寒かったですか。


（ここに答えを書く）


62番はシドニーで、緯度は負（南半球）です。1月がいちばん暖かく、7月がいちばん寒くなります。同じ12個の列でも、南半球では季節が逆に並びます。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| JSONを読む（1節） | 67都市×24列の表にできた（気温12か月・降水量12か月） |
| 単位で変わる？（2節） | ？（このあとの段で） |
| 軸を読む（3節） | ？（このあとの段で） |
| 地図を読む（4節） | ？（このあとの段で） |
| 何本残す？（5節） | ？（このあとの段で） |

まだ分からないのは、24列をまとめた向きが何になるか、です。


---


## 2. 単位で答えは変わるか？


ここから、段2「単位で変わる？」（2節）です。問いは、気温（°C）と降水量（mm）のように単位の違う列で、主成分分析の答えは変わるか、です。

JSONを、キーと番号でたどり、67都市×24列の表にしました。気温は°C、降水量はmmで、単位が違います。このまま主成分分析をかけてみます。


### お手本：列の散らばり


In [ ]:
print(X.std().iloc[[0, 6, 12, 18]].round(1))


列ごとの標準偏差（散らばり）を、1月と7月の気温と降水量で見ました。降水量の散らばりは、気温の約10倍です。


### お手本：そのまま主成分分析

- 役割：標準化せずに、24列に`PCA`を当てはめる（第11回）
- 入力：表`X`
- 出力：`pca_raw`と、上の2本の寄与率（proportion of variance explained）


In [ ]:
from sklearn.decomposition import PCA
pca_raw = PCA()
pca_raw.fit(X)
print(pca_raw.explained_variance_ratio_[:2].round(3))


#### 1行ずつ読む

- 1〜3行目：主成分分析の道具を呼び出し、標準化せずに当てはめる
- 4行目：上の2本の寄与率


### お手本：第1の向きの重み


In [ ]:
plt.bar(range(24), pca_raw.components_[0])
plt.xlabel('列（左12本が気温、右12本が降水量）')
plt.ylabel('重み')
plt.show()


In [ ]:
print((pca_raw.components_[0][:12] ** 2).sum().round(3))


1本の向きの重み（weight）は、2乗して足すと1です。そのうち、気温12本の分は0.001でした。えっ、第1の向きは、ほとんど降水量だけです。気温は、ないのと同じです。

主成分分析は、散らばりが最大の向きを探します（第11回）。散らばりの大きい列（mm）が、向きを引っぱります。単位を替えると、散らばりの大きさが変わり、答えも変わります。


### お手本：標準化して主成分分析

列ごとに、平均を引いて、標準偏差（standard deviation）で割ります（第3回）。どの列も平均0・標準偏差1になり、単位が消えます。単位も意味も違う列が並ぶときは、標準化してから主成分分析をかけます。

- 役割：24列を標準化し、`PCA`を当てはめる
- 入力：表`X`
- 出力：配列`Z`（67×24）と`pca`


In [ ]:
from sklearn.preprocessing import StandardScaler
Z = StandardScaler().fit_transform(X)
pca = PCA()
pca.fit(Z)
print(pca.explained_variance_ratio_[:4].round(3))


#### 1行ずつ読む

- 1・2行目：24列を標準化して、配列`Z`にする
- 3・4行目：`Z`に`PCA`を当てはめる
- 5行目：上の4本の寄与率。1本に偏らず、何本かに分かれた


In [ ]:
plt.bar(range(24), pca.components_[0])
plt.xlabel('列（左12本が気温、右12本が降水量）')
plt.ylabel('重み')
plt.show()


標準化すると、第1の向きには、気温の重みが大きく、降水量の重みも入りました。


> 重要：主成分分析は、散らばりの大きい列に引っぱられる。単位も意味も違う列が並ぶときは、標準化してから分析する。


### TODO②：降水量をcmにする（予想 → 実行 → 比べる）

下のセルは、標準化せずに、右の12列（降水量）に数を掛けてから`PCA`をかけ、第1の向きの重みのうち気温の分を表示します。`.copy()`で写しを作るので、`X`は壊れません。

まず、予想を書いてください。降水量をcmにする（0.1を掛ける）と、第1の向きの気温の分はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`1`を`0.1`に替えて実行してください。


In [ ]:
# TODO: 1 を 0.1 に替えて実行する（mm を cm に）
X_x = X.copy()
X_x.iloc[:, 12:] = X_x.iloc[:, 12:] * 1
p_x = PCA()
p_x.fit(X_x)
print((p_x.components_[0][:12] ** 2).sum().round(3))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。気温の分は、どう変わりましたか。


（ここに答えを書く）


cmにすると、第1の向きの気温の分は0.001から0.839になります。今度は気温が向きを引っぱります。同じデータなのに、単位を替えるだけで、第1の向きの中身が入れ替わります。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| JSONを読む（1節） | 67都市×24列の表にできた（気温12か月・降水量12か月） |
| 単位で変わる？（2節） | 標準化しないと、第1の向きは降水量ばかり（気温の分0.001） |
| 軸を読む（3節） | ？（このあとの段で） |
| 地図を読む（4節） | ？（このあとの段で） |
| 何本残す？（5節） | ？（このあとの段で） |

まだ分からないのは、標準化した向きが、それぞれ何を表すか、です。


---


## 3. 軸は何を表す？


ここから、段3「軸を読む」（3節）です。問いは、標準化して求めた向き（軸）は、それぞれ何を表すか、です。

標準化しないと、第1の向きは降水量ばかりで、cmにすると気温ばかりでした。単位がまちまちなので、標準化してから主成分分析をかけました。寄与率は0.413・0.248・0.165・0.104です。


### 負荷量

向きの24個の数を、負荷量といいます。得点を作るときの、各列の重みです。重みが大きい列ほど、その軸の得点に効きます。列に名前があるので、どの列の重みが大きいかで、軸の意味を読めます。

都市$i$の標準化した24個の値を$z_{i1}, \dots, z_{i,24}$、第$k$の向きの負荷量を$w_{k1}, \dots, w_{k,24}$とすると、得点は次の式です。

$$
t_{ik} = w_{k1} z_{i1} + w_{k2} z_{i2} + \cdots + w_{k,24} z_{i,24}
$$


### お手本：負荷量を月の折れ線に

- 役割：第1の向きの負荷量を、気温12個と降水量12個に分けて、月の折れ線で描く
- 入力：`pca.components_`
- 出力：図


In [ ]:
W = pca.components_
months = range(1, 13)
k = 0
plt.plot(months, W[k][:12], 'o-', label='気温')
plt.plot(months, W[k][12:], 's-', label='降水')
plt.axhline(0, color='gray')
plt.legend()
plt.xlabel('月')
plt.ylabel('負荷量（重み）')
plt.show()


#### 1行ずつ読む

- 1行目：向きを並べた配列（24本×24列）。`W[0]`が第1の向き
- 2行目：月の番号1〜12
- 3行目：描く向きの番号（0が第1の向き）
- 4・5行目：前の12個（気温）と後ろの12個（降水量）を、月ごとの折れ線にする
- 6行目：重み0の横線を引く
- 7〜10行目：凡例と軸の名前を付けて表示する

第1の向きは、気温の重みが12か月すべてで大きな正です。1年中暑い都市ほど、得点が大きくなります。降水量の重みも少し正で、暑い都市は雨も多いのです。


### お手本：年平均気温と比べる


In [ ]:
S = pca.transform(Z)
mean_t = X.iloc[:, :12].mean(axis=1)
print(np.corrcoef(S[:, 0], mean_t)[0, 1].round(3))


#### 1行ずつ読む

- 1行目：67都市の得点（`transform`）
- 2行目：12か月の気温の平均（年平均気温）
- 3行目：第1の得点と、年平均気温の相関

0.951です。第1の向きは「暑さ」と名前を付けてよいでしょう。


### 第2・第3の向き


In [ ]:
k = 1
plt.plot(months, W[k][:12], 'o-', label='気温')
plt.plot(months, W[k][12:], 's-', label='降水')
plt.axhline(0, color='gray')
plt.legend()
plt.xlabel('月')
plt.ylabel('負荷量（重み）')
plt.show()


第2の向きは、降水量の重みが12〜4月で大きな正、6〜8月の気温が負です。12〜4月に雨が多く、6〜8月が寒い都市ほど、得点が大きくなります。


In [ ]:
k = 2
plt.plot(months, W[k][:12], 'o-', label='気温')
plt.plot(months, W[k][12:], 's-', label='降水')
plt.axhline(0, color='gray')
plt.legend()
plt.xlabel('月')
plt.ylabel('負荷量（重み）')
plt.show()


第3の向きは、降水量の重みが6〜9月で大きな正です。夏の雨季（モンスーン）の強さです。


### お手本：得点を手で出す


In [ ]:
print(X.index[0])
print((Z[0] @ W[0]).round(2), S[0, 0].round(2))


#### 1行ずつ読む

- 1行目：0番目の都市（東京）
- 2行目：標準化した24個`@`第1の向きの24個（重み付きの和）と、`transform`の得点

同じ値です。東京は、暑さの平均（0）より少し暑い側です。


> 今日の重要概念（統計）：軸は負荷量で読む
>
> 主成分（principal component）の軸は、負荷量（得点を作るときの各列の重み）で読み、名前を付ける。単位がまちまちな列は、標準化してから読む。


### TODO③：第4の向きを描く（予想 → 実行 → 比べる）

下のセルは、第`k + 1`の向きの負荷量を、月の折れ線で描きます。

まず、予想を書いてください。第4の向き（番号は3）は、どんな折れ線になるでしょうか。何と名前を付けますか。


（ここに予想を書く）


予想を書いたら、セルの`0`を`3`に替えて実行してください。


In [ ]:
# TODO: 0 を 3 に替えて実行する
k = 0
plt.plot(months, W[k][:12], 'o-', label='気温')
plt.plot(months, W[k][12:], 's-', label='降水')
plt.axhline(0, color='gray')
plt.legend()
plt.xlabel('月')
plt.ylabel('負荷量（重み）')
plt.show()


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。どんな折れ線になりましたか。何と名前を付けましたか。


（ここに答えを書く）


気温の重みが、7月に正、1月に負になります。7月が暖かく1月が寒い都市ほど、得点が大きくなります。季節の向きで、北半球らしさと名前を付けられます。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| JSONを読む（1節） | 67都市×24列の表にできた（気温12か月・降水量12か月） |
| 単位で変わる？（2節） | 標準化しないと、第1の向きは降水量ばかり（気温の分0.001） |
| 軸を読む（3節） | 第1は暑さ、第2は12〜4月の雨、第3は6〜9月の雨 |
| 地図を読む（4節） | ？（このあとの段で） |
| 何本残す？（5節） | ？（このあとの段で） |

まだ分からないのは、都市が地図の上でどう並ぶか、です。


---


## 4. 都市はどこに並ぶ？


ここから、段4「地図を読む」（4節）です。問いは、得点を地図にすると、都市はどう並ぶか、です。

第1の向きは暑さ、第2は12〜4月の雨、第3は6〜9月の雨、第4は季節の向きでした。負荷量で軸を読み、名前を付けました。ここから、67都市の得点を地図にします。


### お手本：得点の地図

- 役割：第1と第2の得点で都市を点で打ち、名前を書く
- 入力：得点`S`、表`X`の行の名前
- 出力：図


In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(S[:, 0], S[:, 1])
for i in range(len(X)):
    plt.text(S[i, 0], S[i, 1], X.index[i], fontsize=8)
plt.xlabel('第1の得点（暑さ）')
plt.ylabel('第2の得点（12〜4月の雨）')
plt.show()


#### 1行ずつ読む

- 1行目：名前が重ならないように、図を大きくする
- 2行目：横に第1、縦に第2の得点をとり、都市を点で打つ
- 3・4行目：都市ごとに、点の位置に名前を書く（`plt.text`）
- 5〜7行目：軸の名前を付けて表示する

右ほど暑く（シンガポール・ムンバイ）、左ほど寒い（ヤクーツク・モスクワ）都市です。上ほど12〜4月に雨が多く、下は乾いた都市（ドバイ・カイロ）です。上のほうには、どんな都市が集まっているでしょうか。


### お手本：南北で得点を比べる

- 役割：都市の緯度をためて、南半球かどうかの配列を作り、第2の得点の平均を南北で比べる
- 入力：`data['cities']`、得点`S`
- 出力：配列`south`と表示


In [ ]:
lats = []
for city in data['cities']:
    lats.append(city['response']['latitude'])
south = np.array(lats) < 0
print(south.sum())
print(S[south, 1].mean().round(2), S[~south, 1].mean().round(2))


#### 1行ずつ読む

- 1〜3行目：都市の辞書から、APIの返事の緯度をたどってためる
- 4行目：緯度が負（南半球）ならTrueの配列`south`
- 5・6行目：南半球の数と、第2の得点の平均（南・北）。`~south`は、TrueとFalseを入れ替えたもの（北半球）

えっ、南半球は2.1、北半球は−0.71です。緯度は1列も使っていないのに、南半球の都市が、第2の向きで上に集まります。


In [ ]:
plt.scatter(S[~south, 0], S[~south, 1], label='北半球')
plt.scatter(S[south, 0], S[south, 1], label='南半球')
plt.legend()
plt.xlabel('第1の得点（暑さ）')
plt.ylabel('第2の得点（12〜4月の雨）')
plt.show()


### なぜ南半球が上に？

南半球では、12〜2月が夏です。夏の雨が、12〜4月の雨になります。また、6〜8月は冬で、6〜8月の気温の重みは負なので、得点を上げます。第2の向きは、「南半球の夏の雨」をとらえていたのです。


> 重要：負荷量で付けた軸の名前は、地図の上の都市を見て確かめる。使わなかった情報（緯度）と並べると、名前の意味が深まる。


### TODO④：縦軸を第4の向きに（予想 → 実行 → 比べる）

下のセルは、北半球と南半球を色分けして、横に第1、縦に第`ky + 1`の得点をとった地図を描きます。

まず、予想を書いてください。縦軸を季節の向き（第4、番号は3）にすると、南北はどう分かれるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`1`を`3`に替えて実行してください。


In [ ]:
# TODO: 1 を 3 に替えて実行する
ky = 1
plt.scatter(S[~south, 0], S[~south, ky], label='北半球')
plt.scatter(S[south, 0], S[south, ky], label='南半球')
plt.legend()
plt.xlabel('第1の得点')
plt.ylabel('縦軸の得点')
plt.show()


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。南北は、どう分かれましたか。


（ここに答えを書く）


南半球はほぼ下、北半球は上に分かれます。下のセルで、第4の得点と緯度の相関が0.656になることを確かめられます。季節の向きは、緯度とよく対応しています。


In [ ]:
print(np.corrcoef(S[:, 3], lats)[0, 1].round(3))


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| JSONを読む（1節） | 67都市×24列の表にできた（気温12か月・降水量12か月） |
| 単位で変わる？（2節） | 標準化しないと、第1の向きは降水量ばかり（気温の分0.001） |
| 軸を読む（3節） | 第1は暑さ、第2は12〜4月の雨、第3は6〜9月の雨 |
| 地図を読む（4節） | 緯度を使わずに、第2の向きで南半球の都市が上に集まる |
| 何本残す？（5節） | ？（このあとの段で） |

まだ分からないのは、2本の地図を、どこまで信じてよいか、です。


---


## 5. 何本残す？


ここから、段5「何本残す？」（5節）です。問いは、2本の地図は、24列の都市の違いを、どこまで写しているか、です。

第1×第2の地図で、暑さと12〜4月の雨が読め、南半球が上に集まりました。第4の向きにも、季節という意味がありました。2本に縮めて、失ったものはどれくらいでしょうか。


### お手本：累積寄与率


In [ ]:
cum = pca.explained_variance_ratio_.cumsum()
print(cum[:5].round(3))
print(pca.explained_variance_[:6].round(2))


#### 1行ずつ読む

- 1・2行目：寄与率を前から足し上げる（第11回）
- 3行目：向きごとの得点の分散（固有値）。標準化したので、列1本ぶんが1

2本で0.661、4本で0.93です。固有値（eigenvalue）が1より大きい向き（列1本より多くを持つ向き）は4本で、累積寄与率が9割を超えるのも4本です。


In [ ]:
plt.bar(range(1, 11), pca.explained_variance_ratio_[:10], label='寄与率')
plt.plot(range(1, 11), cum[:10], 'o-', color='tab:red', label='累積寄与率')
plt.xlabel('向きの本数')
plt.legend()
plt.show()


### 地図の距離は、元の距離を写すか

地図で近い都市は、24列でも似ているはずです。24列での都市どうしの距離と、地図での距離を、すべての組で比べます。67都市なら、2,211組です。


### お手本：距離を比べる

- 役割：すべての組の距離を、24列と2本の地図で出し、相関を見る
- 入力：`Z`、得点`S`
- 出力：組の数と、2つの距離の相関係数


In [ ]:
from scipy.spatial.distance import pdist
d24 = pdist(Z)
d2 = pdist(S[:, :2])
print(len(d24), np.corrcoef(d24, d2)[0, 1].round(3))


#### 1行ずつ読む

- 1・2行目：都市のすべての組の距離を出す（`pdist`）。標準化した24列で
- 3行目：同じ組の、2本の地図での距離
- 4行目：組の数と、2つの距離の相関係数


In [ ]:
plt.scatter(d24, d2, s=3, alpha=0.3)
plt.plot([0, d24.max()], [0, d24.max()], color='gray')
plt.xlabel('24列での距離')
plt.ylabel('2本の地図での距離')
plt.show()


相関は0.919です。地図の距離は、元の距離より短く出ます（点は斜めの線より下）。縮めた分だけ、近く見えます。


> 重要：2本の地図は、24列の近似にすぎない。説明に使う前に、累積寄与率と、元の距離をどれだけ写せているかを確かめる。


### TODO⑤：本数を4にする（予想 → 実行 → 比べる）

下のセルは、上の何本かの得点で距離を測り、24列での距離との相関を出します。

まず、予想を書いてください。4本の得点で距離を測ると、相関はどこまで上がるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`2`を`4`に替えて実行してください。


In [ ]:
# TODO: 2 を 4 に替えて実行する
d_k = pdist(S[:, :2])
print(np.corrcoef(d24, d_k)[0, 1].round(3))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。相関は、どこまで上がりましたか。


（ここに答えを書く）


4本では0.995です。都市どうしの違いは、4本の向きでほぼ写せます。地図（2本）に描けない分は、第3・第4の向き（夏の雨、季節）にあります。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| JSONを読む（1節） | 67都市×24列の表にできた（気温12か月・降水量12か月） |
| 単位で変わる？（2節） | 標準化しないと、第1の向きは降水量ばかり（気温の分0.001） |
| 軸を読む（3節） | 第1は暑さ、第2は12〜4月の雨、第3は6〜9月の雨 |
| 地図を読む（4節） | 緯度を使わずに、第2の向きで南半球の都市が上に集まる |
| 何本残す？（5節） | 2本で累積寄与率0.661、距離の相関0.919。4本なら0.93 |

5つの段がそろい、今日の問いに答えられます。


### 答え

気候の24個の数は、暑さ・12〜4月の雨・6〜9月の雨・季節の4本で93%を持てます。第1×第2の地図では、暑さと南半球の夏の雨が読め、緯度を使わずに南半球が分かれました。ただし、標準化しないと降水量ばかりの向きになります。単位で答えが変わります。


まだ言えないこと

- 軸の名前は、読み手の解釈。同じ負荷量に、別の名前を付ける人もいる
- 10年の平均だけを見た。年ごとの揺れや、気候の変化は見ていない
- 67都市の選び方で、向きは変わりうる


### レポートなら、こう書く

今日の解析を、期末レポートの項目に写すと、次のようになります。

- データ：Open-Meteoから67都市の2015〜2024年の毎日の値を取り、月ごとの気温と降水量（24列）にした
- 前処理（preprocessing）：標準化しないと第1の向きは降水量ばかり（気温の分0.001）なので、標準化した
- 軸：第1は暑さ（年平均気温と相関0.951）、第2は12〜4月の雨、第3は6〜9月の雨
- 驚き：緯度を使わずに、第2の向きで南半球の都市が上に集まった（南半球の夏の雨）
- 本数：2本で累積寄与率0.661、4本で0.93。距離の相関は2本0.919、4本0.995
- 限界：軸の名前は解釈。10年の平均で、都市の選び方にも左右される


---


## 目標に答えられたか


- 今日の目標は「世界の都市の気候を少ない軸にまとめると、その軸は何を表すか。地図の上で都市はどう並ぶか」でした。負荷量の折れ線と、地図と、累積寄与率で答えられますか？
- 1節で、東京の毎日の気温のリストに届くまで、何回、どんなキーと番号でたどりましたか？
- 2節で、標準化しないと気温が向きに入らなかったのはなぜですか？
- 3節で、第2の向きに「12〜4月の雨」と名前を付けたのは、負荷量のどこを見たからですか？
- 4節で、緯度を使わないのに南半球が分かれたのはなぜですか？
- 5節で、2本の地図の距離が、元の距離より短く出るのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：単位がまちまちな列は、標準化してから主成分分析をかける。軸は負荷量で読んで名前を付け、地図で確かめる。地図は近似なので、累積寄与率と距離の写り方を確かめてから説明に使う
- 重要概念（統計）：軸は負荷量で読む。主成分の軸は、負荷量（得点を作るときの各列の重み）で読み、名前を付ける。単位がまちまちな列は、標準化してから読む
- 重要概念（Python）：入れ子の辞書とリスト。JSONは、辞書とリストの入れ子。キーと番号を、外から順にたどると、欲しい値に届く
- 主成分分析は、散らばりの大きい列に引っぱられる。単位で答えが変わる
- 地図は近似。累積寄与率と距離の相関で、どこまで写せているかを確かめる


---


## 次回へ


今日は、数の列を少ない軸にまとめました。では、数えた表（何がいくつ、の組み合わせ）は、どう地図にすればよいでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：軸は負荷量で読む。主成分の軸は、負荷量（得点を作るときの各列の重み）で読み、名前を付ける。単位がまちまちな列は、標準化してから読む（3節）
- Python：入れ子の辞書とリスト。JSONは、辞書とリストの入れ子。キーと番号を、外から順にたどると、欲しい値に届く（1節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

軸は負荷量で読む、なら、たとえば「料理の味の軸」を描けます。
料理の味を、いくつかの材料の量で表します。「辛さの軸」は、唐辛子と胡椒の重みが大きく、砂糖の重みが負です。重み（負荷量）を見れば、軸に名前が付きます。料理の点数（得点）は、材料の量の重み付きの和です。3節の負荷量の折れ線が手本になります。

</details>


<details><summary>詰まったら</summary>

入れ子の辞書とリストなら、たとえば「住所をたどって家に届く」を描けます。
国 → 都道府県 → 市 → 番地と、外から順にたどります。大きな箱（辞書）を開けると引き出しの列（リスト）があり、その1段を開けると、また箱があります。1節の、1回ずつたどる例が手本になります。

</details>


---


## 発展（任意）


### 別の地図：t-SNE

t-SNEは、近い都市どうしが地図でも近くなるように、点を並べ直す方法です。主成分分析のような、向きや重みはありません。同じ67都市で地図を作り、元の距離との相関を見ます。


In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE(2, perplexity=10, random_state=0, init='pca')
E = tsne.fit_transform(Z)
print(E.shape)
print(np.corrcoef(d24, pdist(E))[0, 1].round(3))


#### 1行ずつ読む

- 1行目：t-SNEの道具を呼び出す
- 2・3行目：2次元の地図にする。`perplexity`は、何都市くらいを近所とみなすか
- 4行目：形は67都市×2
- 5行目：元の距離との相関


In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(E[~south, 0], E[~south, 1], label='北半球')
plt.scatter(E[south, 0], E[south, 1], label='南半球')
for i in range(len(X)):
    plt.text(E[i, 0], E[i, 1], X.index[i], fontsize=8)
plt.legend()
plt.show()


相関は0.604で、主成分分析の2本（0.919）より低くなります。t-SNEは近所の並びを残しますが、遠い都市どうしの距離は残しません。軸にも意味がありません。

試すなら、`perplexity`を30にして、相関がどう変わるかを比べてみましょう。UMAPという方法もあります（Colabでは`pip install umap-learn`が要ります）。
